# MultiCaRe Multimodal Diagnosis Support — Reproducible Pipeline

Clean end-to-end notebook for the final project pipeline. It intentionally removes exploratory disease discovery, semantic-validation pilots, Qwen generation, and failed early-fusion training from the executable path.

**Final classes:** Aspergillosis, Herpes simplex, Syphilis, Tuberculosis.

**Input:** `../data/processed/final_training_dataset.csv` containing verified MultiCaRe case-image pairs and the leakage-safe split.

Pipeline: **ResNet18 image baseline → MiniLM/FAISS text retrieval → ResNet image retrieval → Late Fusion → weighted RRF → diagnosis + retrieved evidence.**

> Academic prototype only; not a clinically validated diagnostic system.


## 1. Install dependencies


In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib pillow torch torchvision sentence-transformers faiss-cpu


## 2. Imports and configuration


In [ ]:
from pathlib import Path
from collections import Counter
import random, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights

from sentence_transformers import SentenceTransformer
import faiss
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT = Path("..")
DATA_FILE = ROOT/"data"/"processed"/"final_training_dataset.csv"
RESULT_DIR, MODEL_DIR, INDEX_DIR = ROOT/"results", ROOT/"models", ROOT/"indexes"
for p in (RESULT_DIR, MODEL_DIR, INDEX_DIR): p.mkdir(parents=True, exist_ok=True)

CLASSES = ["Aspergillosis","Herpes simplex","Syphilis","Tuberculosis"]
label2id = {x:i for i,x in enumerate(CLASSES)}
id2label = {i:x for x,i in label2id.items()}
TEXT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 5
LATE_ALPHA = 0.5
RRF_TEXT_WEIGHT, RRF_IMAGE_WEIGHT, RRF_K = 1.0, 0.25, 60
print("Device:", device)


## 3. Load curated MultiCaRe data

The discovery/cleaning stage produced 320 balanced cases and 643 verified images. For the compact case-level experiment, one representative image is selected per case. The stored `split` must be article-level: 221 train / 48 validation / 51 test.


In [ ]:
df = pd.read_csv(DATA_FILE)
required = {"article_id","case_id","case_text","disease","split","image_path"}
assert not (required-set(df.columns)), f"Missing columns: {required-set(df.columns)}"
df = df[df.disease.isin(CLASSES)].copy()

def resolve_path(x):
    p = Path(str(x))
    return str(p if p.exists() else ROOT/p)
df["image_path"] = df["image_path"].map(resolve_path)

case_df = (df.sort_values(["article_id","case_id","image_path"])
             .drop_duplicates(["article_id","case_id"]).reset_index(drop=True))
assert len(case_df) == 320
assert case_df.image_path.map(lambda x: Path(x).exists()).all()

train_df = case_df[case_df.split=="train"].reset_index(drop=True)
val_df   = case_df[case_df.split=="val"].reset_index(drop=True)
test_df  = case_df[case_df.split=="test"].reset_index(drop=True)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print(case_df.groupby(["split","disease"]).size())


## 4. Leakage check


In [ ]:
A, B, C = set(train_df.article_id), set(val_df.article_id), set(test_df.article_id)
assert A.isdisjoint(B) and A.isdisjoint(C) and B.isdisjoint(C)
print("Article leakage: 0")


## 5. Train image-only ResNet18 baseline


In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((224,224)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10), transforms.ColorJitter(brightness=.1, contrast=.1),
    transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])
])
eval_tf = transforms.Compose([
    transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])
])

class ImgDS(Dataset):
    def __init__(self, frame, tf): self.frame, self.tf = frame.reset_index(drop=True), tf
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        r = self.frame.iloc[i]
        return self.tf(Image.open(r.image_path).convert("RGB")), label2id[r.disease]

train_loader = DataLoader(ImgDS(train_df,train_tf),batch_size=16,shuffle=True)
val_loader = DataLoader(ImgDS(val_df,eval_tf),batch_size=16)
test_loader = DataLoader(ImgDS(test_df,eval_tf),batch_size=16)

image_model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
for p in image_model.parameters(): p.requires_grad=False
n = image_model.fc.in_features
image_model.fc = nn.Sequential(nn.Dropout(.3),nn.Linear(n,len(CLASSES)))
image_model = image_model.to(device)

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(image_model.fc.parameters(),lr=1e-3,weight_decay=1e-4)
ckpt = MODEL_DIR/"best_resnet18_image_only.pt"
best, bad, patience = -1, 0, 4

for epoch in range(10):
    image_model.train()
    tr_ok=tr_n=0
    for x,y in train_loader:
        x,y=x.to(device),y.to(device); opt.zero_grad()
        z=image_model(x); loss=loss_fn(z,y); loss.backward(); opt.step()
        tr_ok += (z.argmax(1)==y).sum().item(); tr_n += len(y)

    image_model.eval(); va_ok=va_n=0
    with torch.no_grad():
        for x,y in val_loader:
            z=image_model(x.to(device)); y=y.to(device)
            va_ok += (z.argmax(1)==y).sum().item(); va_n += len(y)
    va=va_ok/va_n
    print(f"Epoch {epoch+1:02d} | train_acc={tr_ok/tr_n:.4f} | val_acc={va:.4f}")
    if va>best:
        best,bad=va,0; torch.save(image_model.state_dict(),ckpt)
    else:
        bad+=1
        if bad>=patience: break

image_model.load_state_dict(torch.load(ckpt,map_location=device)); image_model.eval()


In [ ]:
def eval_image_classifier():
    yt,yp=[],[]
    with torch.no_grad():
        for x,y in test_loader:
            p=image_model(x.to(device)).argmax(1).cpu().tolist()
            yt += y.tolist(); yp += p
    return yt,yp
img_true,img_pred=eval_image_classifier()
image_acc=accuracy_score(img_true,img_pred)
image_f1=f1_score(img_true,img_pred,average="macro")
print(f"Image-only accuracy={image_acc:.4f}, macro-F1={image_f1:.4f}")


## 6. MiniLM + FAISS text retrieval (TRAIN cases only)


In [ ]:
def chunks(text,size=150,overlap=30):
    w=str(text).split(); out=[]; s=0
    while s<len(w):
        e=s+size; out.append(" ".join(w[s:e]))
        if e>=len(w): break
        s=e-overlap
    return out

text_encoder=SentenceTransformer(TEXT_MODEL,device=str(device))
texts,meta=[],[]
for r in train_df.itertuples():
    for j,c in enumerate(chunks(r.case_text)):
        texts.append(c); meta.append({"case_id":r.case_id,"article_id":r.article_id,"disease":r.disease,"chunk_idx":j})
meta=pd.DataFrame(meta)
emb=text_encoder.encode(texts,batch_size=32,normalize_embeddings=True,show_progress_bar=True).astype("float32")
text_index=faiss.IndexFlatIP(emb.shape[1]); text_index.add(emb)
faiss.write_index(text_index,str(INDEX_DIR/"text_faiss.index"))
meta.to_csv(INDEX_DIR/"text_faiss_metadata.csv",index=False)

def retrieve_text(q,k=5,search_k=30):
    qe=text_encoder.encode([str(q)],normalize_embeddings=True).astype("float32")
    scores,ids=text_index.search(qe,min(search_k,text_index.ntotal))
    best={}
    for idx,score in zip(ids[0],scores[0]):
        m=meta.iloc[int(idx)]; cid=m.case_id
        if cid not in best or score>best[cid]["score"]:
            best[cid]={"case_id":cid,"article_id":m.article_id,"disease":m.disease,"score":float(score)}
    return sorted(best.values(),key=lambda x:x["score"],reverse=True)[:k]

def majority(rs): return Counter(r["disease"] for r in rs).most_common(1)[0][0]

def retrieval_metrics(frame,fn,k=5):
    top1=hit=0; rr=[]; yt=[]; yp=[]
    for r in frame.itertuples():
        rs=fn(r,k); ds=[x["disease"] for x in rs]
        yt.append(r.disease); yp.append(majority(rs)); top1 += ds[0]==r.disease
        if r.disease in ds: hit+=1; rr.append(1/(ds.index(r.disease)+1))
        else: rr.append(0)
    return {"top1":top1/len(frame),"recall@5":hit/len(frame),"mrr@5":np.mean(rr),
            "majority_acc":accuracy_score(yt,yp),"majority_f1":f1_score(yt,yp,average="macro")}

text_metrics=retrieval_metrics(test_df,lambda r,k:retrieve_text(r.case_text,k),TOP_K)
text_metrics


## 7. ResNet18 image retrieval (TRAIN cases only)


In [ ]:
image_encoder=nn.Sequential(*list(image_model.children())[:-1]).to(device).eval()
def encode_image(path):
    x=eval_tf(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad(): z=F.normalize(image_encoder(x).flatten(1),p=2,dim=1)
    return z.cpu().numpy()[0].astype("float32")

im_emb=np.vstack([encode_image(p) for p in train_df.image_path]).astype("float32")
image_index=faiss.IndexFlatIP(im_emb.shape[1]); image_index.add(im_emb)
faiss.write_index(image_index,str(INDEX_DIR/"image_faiss.index"))
im_meta=train_df[["article_id","case_id","disease","image_path"]].reset_index(drop=True)
im_meta.to_csv(INDEX_DIR/"image_faiss_metadata.csv",index=False)

def retrieve_image(path,k=5):
    scores,ids=image_index.search(encode_image(path).reshape(1,-1),k); out=[]
    for rank,(idx,score) in enumerate(zip(ids[0],scores[0]),1):
        r=im_meta.iloc[int(idx)]
        out.append({"rank":rank,"score":float(score),"case_id":r.case_id,"article_id":r.article_id,"disease":r.disease})
    return out

image_ret_metrics=retrieval_metrics(test_df,lambda r,k:retrieve_image(r.image_path,k),TOP_K)
image_ret_metrics


## 8. Late decision fusion


In [ ]:
def image_probs(path):
    x=eval_tf(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad(): p=F.softmax(image_model(x),dim=1)[0].cpu().numpy()
    return {id2label[i]:float(p[i]) for i in range(len(CLASSES))}

def text_scores(text,k=5):
    c=Counter(x["disease"] for x in retrieve_text(text,k))
    return {d:c.get(d,0)/k for d in CLASSES}

def late_predict(r,alpha=LATE_ALPHA):
    t,i=text_scores(r.case_text),image_probs(r.image_path)
    s={d:alpha*t[d]+(1-alpha)*i[d] for d in CLASSES}
    return max(s,key=s.get)

late_true=test_df.disease.tolist()
late_pred=[late_predict(r) for r in test_df.itertuples()]
late_acc=accuracy_score(late_true,late_pred); late_f1=f1_score(late_true,late_pred,average="macro")
print(f"Late fusion accuracy={late_acc:.4f}, macro-F1={late_f1:.4f}")


## 9. Final multimodal retrieval — weighted RRF

Validation-selected settings: text weight **1.0**, image weight **0.25**, RRF constant **60**. Do not retune these on the test set.


In [ ]:
def rrf(text_rs,image_rs):
    scores={}; info={}
    for rank,x in enumerate(text_rs,1):
        cid=x["case_id"]; scores[cid]=scores.get(cid,0)+RRF_TEXT_WEIGHT/(RRF_K+rank); info[cid]=x
    for rank,x in enumerate(image_rs,1):
        cid=x["case_id"]; scores[cid]=scores.get(cid,0)+RRF_IMAGE_WEIGHT/(RRF_K+rank); info[cid]=x
    return [{**info[cid],"rrf_score":s} for cid,s in sorted(scores.items(),key=lambda x:x[1],reverse=True)]

def multimodal_retrieve(text,image_path,k=5,search_k=20):
    return rrf(retrieve_text(text,search_k,max(30,search_k)),retrieve_image(image_path,search_k))[:k]

def final_diagnose(text,image_path,k=5):
    rs=multimodal_retrieve(text,image_path,k)
    counts=Counter(x["disease"] for x in rs); pred,votes=counts.most_common(1)[0]
    agreement=votes/len(rs)
    strength="Strong" if agreement>=.8 else "Moderate" if agreement>=.6 else "Weak"
    conflicts={d:n for d,n in counts.items() if d!=pred}
    summary=f"{votes} of {len(rs)} retrieved MultiCaRe cases support {pred}."
    if conflicts: summary+=" Conflicting retrieved labels: "+", ".join(f"{d} ({n})" for d,n in conflicts.items())+"."
    return {"prediction":pred,"retrieval_agreement":agreement,"evidence_strength":strength,
            "votes":dict(counts),"evidence_summary":summary,"retrieved_cases":rs}

rrf_true=[]; rrf_pred=[]; records=[]
for r in test_df.itertuples():
    z=final_diagnose(r.case_text,r.image_path)
    rrf_true.append(r.disease); rrf_pred.append(z["prediction"])
    records.append({"case_id":r.case_id,"true_label":r.disease,"prediction":z["prediction"],
                    "correct":r.disease==z["prediction"],"votes":str(z["votes"])})
rrf_acc=accuracy_score(rrf_true,rrf_pred); rrf_f1=f1_score(rrf_true,rrf_pred,average="macro")
print(f"RRF accuracy={rrf_acc:.4f}, macro-F1={rrf_f1:.4f}")
print(classification_report(rrf_true,rrf_pred,labels=CLASSES,digits=4,zero_division=0))


## 10. Save outputs


In [ ]:
cm=confusion_matrix(rrf_true,rrf_pred,labels=CLASSES)
disp=ConfusionMatrixDisplay(cm,display_labels=CLASSES)
fig,ax=plt.subplots(figsize=(8,6)); disp.plot(ax=ax,xticks_rotation=45)
plt.title("Multimodal Retrieval + Weighted RRF"); plt.tight_layout()
plt.savefig(RESULT_DIR/"rrf_confusion_matrix.png",dpi=300,bbox_inches="tight"); plt.show()

summary=pd.DataFrame([
    ["Image-only ResNet18",image_acc,image_f1],
    ["Text FAISS Top-5 Majority",text_metrics["majority_acc"],text_metrics["majority_f1"]],
    ["Image Retrieval Top-5 Majority",image_ret_metrics["majority_acc"],image_ret_metrics["majority_f1"]],
    ["Late Fusion Image + Text",late_acc,late_f1],
    ["Multimodal Retrieval + RRF",rrf_acc,rrf_f1]
],columns=["Method","Accuracy","Macro_F1"])
summary.to_csv(RESULT_DIR/"reproduced_results.csv",index=False)
pd.DataFrame(records).to_csv(RESULT_DIR/"rrf_predictions.csv",index=False)
display(summary)


## Original experiment reference

The original run reported:

| Method | Accuracy | Macro-F1 |
|---|---:|---:|
| Image-only ResNet18 | 29.41% | 0.2224 |
| Image Retrieval Top-5 Majority | 31.37% | 0.2961 |
| Text FAISS Top-1 | 52.94% | — |
| Text FAISS Top-5 Majority | 56.86% | 0.5750 |
| Text FAISS + Qwen2.5-1.5B | 49.02% | 0.4274 |
| Early Fusion ResNet18 + DistilBERT | 13.73% | 0.0816 |
| **Late Fusion Image + Text** | **58.82%** | **0.5913** |
| **Multimodal Retrieval + RRF** | **58.82%** | **0.5820** |

Qwen and early fusion are retained as ablation/negative results in the report, but omitted from this clean executable pipeline.
